# Day 09 · Urban-rural — Where the Grid Frays

**Question.** Chicago's address grid runs on past the city limit, a road every mile from State
Street to the farms. Does transit service follow it? Where does a square mile stop being urban for
someone who cannot drive?

**Approach.** Count, for every 2020 Census block in the six RTA counties, the buses and trains an
hour that stop near it on a weekday (CTA, Metra and Pace GTFS). Sum the blocks into the square
miles of the address grid: residents against service. Then draw the edge of frequent service: where
four or more buses and trains an hour, one every 15 minutes, stop within reach.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | GTFS feeds (CTA, Metra, Pace), 2020 blocks of the six counties, place names |
| 3. Validate | Feed dates cover the service day, trip counts, population totals |
| 4. Service | Departures by stop and route; buses and trains per hour near each block |
| 5. Frequent edge | The same count on a tenth-of-a-mile lattice; the line where it reaches four an hour |
| 6. Analyze | Square miles, the index from the Loop outward, equal density, the inset, the finding |
| 7. QA view | Quick map and tables |
| 8. Export | `plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import io
import json
import logging
import re
import zipfile
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from scipy.spatial import cKDTree
from shapely.geometry import LineString, MultiPolygon, Polygon

from atlas import Ledger, write_plate
from atlas import grid
from atlas.census import GEO, PL_URL, RAW_DIR as CENSUS_RAW

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day09')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
DAY = 9
SERVICE_DAY = date(2026, 11, 10)  # a Tuesday, inside every feed
DOW = 'tuesday'
WINDOW = (7, 19)                  # service counted 7:00-19:00
BUS_R, RAIL_R = 0.25, 0.5         # miles from a block to a stop it can use
FREQ_TPH = 4                      # frequent: four or more an hour each way, one every 15 minutes
EDGE_STEP = 0.1                   # miles between lattice points for the frequent edge
EDGE_MIN = 0.5                    # sq mi: islands and holes smaller than this are not drawn
FULL_TPH = 12                     # an hour's service that fills a square: one every 5 minutes each way
EXTENT = (-56, 6, -48, 44)        # miles west, east, south, north: the plate's map
COUNTIES = {'031': 'Cook', '043': 'DuPage', '089': 'Kane', '097': 'Lake', '111': 'McHenry', '197': 'Will'}
CHICAGO = '14000'

ROOT = Path('../..').resolve()
GTFS_DIR = ROOT / 'reference' / 'raw' / 'gtfs'
DAY_DIR = Path('.').resolve()
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
FEEDS = {
    'cta': {'name': 'CTA', 'glob': 'cta_gtfs_*.zip', 'url': 'https://www.transitchicago.com/developers/gtfs/'},
    'metra': {'name': 'Metra', 'glob': 'metra_gtfs_*.zip', 'url': 'https://metra.com/developers'},
    'pace': {'name': 'Pace', 'glob': 'pace_gtfs_*.zip', 'url': 'https://www.pacebus.com/route-timetable-data-services'},
}

## 2. Pull

GTFS snapshots live in `reference/raw/gtfs/` (gitignored; dated file names). The newest snapshot
of each feed is used. Census blocks come from the P.L. 94-171 bulk file already used since Day 03.

In [ ]:
def newest(pattern: str) -> Path:
    files = sorted(GTFS_DIR.glob(pattern))
    assert files, f'no feed matching {pattern} in {GTFS_DIR}'
    return files[-1]


def read(z: zipfile.ZipFile, name: str, cols: set[str] | None = None) -> pd.DataFrame:
    # Metra's headers carry leading spaces
    with z.open(name) as f:
        df = pd.read_csv(f, dtype=str, skipinitialspace=True, usecols=lambda c: cols is None or c.strip() in cols)
    df.columns = [c.strip() for c in df.columns]
    return df


for key, f in FEEDS.items():
    f['path'] = newest(f['glob'])
    f['snapshot'] = date.fromisoformat(re.search(r'(\d{4}-\d{2}-\d{2})', f['path'].name).group(1))
    log.info('%s: %s', key, f['path'].name)

In [ ]:
D = int(SERVICE_DAY.strftime('%Y%m%d'))


def active_services(z: zipfile.ZipFile) -> set[str]:
    cal = read(z, 'calendar.txt')
    on = set(cal[(cal[DOW] == '1') & (cal['start_date'].astype(int) <= D) & (cal['end_date'].astype(int) >= D)]['service_id'])
    if 'calendar_dates.txt' in z.namelist():
        cd = read(z, 'calendar_dates.txt')
        cd = cd[cd['date'].astype(int) == D]
        on |= set(cd.loc[cd['exception_type'] == '1', 'service_id'])
        on -= set(cd.loc[cd['exception_type'] == '2', 'service_id'])
    return on


def hours(t: pd.Series) -> pd.Series:
    p = t.str.split(':', expand=True).astype(int)
    return p[0] + p[1] / 60 + p[2] / 3600


deps, trips, shapes, stops = [], [], {}, []
for ag, f in FEEDS.items():
    z = zipfile.ZipFile(f['path'])
    on = active_services(z)
    rt = read(z, 'routes.txt', {'route_id', 'route_type', 'route_short_name', 'route_long_name'})
    tr = read(z, 'trips.txt', {'trip_id', 'service_id', 'route_id', 'shape_id'})
    tr = tr[tr['service_id'].isin(on)].merge(rt, on='route_id')
    st = read(z, 'stop_times.txt', {'trip_id', 'departure_time', 'stop_id', 'stop_sequence'})
    st = st[st['trip_id'].isin(set(tr['trip_id']))].copy()
    st['h'] = hours(st['departure_time'])
    st['seq'] = st['stop_sequence'].astype(int)
    sp = read(z, 'stops.txt', {'stop_id', 'stop_name', 'stop_lat', 'stop_lon'})
    e, n = grid.miles_from_origin(sp['stop_lat'].astype(float).to_numpy(), sp['stop_lon'].astype(float).to_numpy())
    sp = sp.assign(e=e, n=n, agency=ag)
    st = st.merge(tr[['trip_id', 'route_id', 'route_type']], on='trip_id').merge(sp[['stop_id', 'e', 'n']], on='stop_id')
    st['agency'] = ag
    span = st.groupby('trip_id')['h'].agg(['min', 'max'])
    tr = tr.join(span, on='trip_id').assign(agency=ag)
    sh = read(z, 'shapes.txt', {'shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence'})
    sh = sh[sh['shape_id'].isin(set(tr['shape_id']))].copy()
    sh['s'] = sh['shape_pt_sequence'].astype(int)
    e, n = grid.miles_from_origin(sh['shape_pt_lat'].astype(float).to_numpy(), sh['shape_pt_lon'].astype(float).to_numpy())
    sh['e'], sh['n'] = e, n
    for sid, g in sh.sort_values(['shape_id', 's']).groupby('shape_id'):
        if len(g) > 1:
            shapes[(ag, sid)] = LineString(g[['e', 'n']].to_numpy())
    f['services'], f['trips'] = len(on), len(tr)
    f['routes'] = tr['route_id'].nunique()
    deps.append(st[['agency', 'trip_id', 'route_id', 'route_type', 'stop_id', 'seq', 'h', 'e', 'n']])
    trips.append(tr)
    stops.append(sp)
    log.info('%s: %d services, %d trips, %d routes, %d stop departures', ag, len(on), len(tr), f['routes'], len(st))
    del st, sh
deps = pd.concat(deps, ignore_index=True)
trips = pd.concat(trips, ignore_index=True)
stops = pd.concat(stops, ignore_index=True)

In [ ]:
# 2020 blocks of the six counties, and place names
rows, places = [], {}
with zipfile.ZipFile(ROOT / CENSUS_RAW / 'il2020.pl.zip') as z, z.open('ilgeo2020.pl') as f:
    for line in io.TextIOWrapper(f, encoding='latin-1'):
        v = line.split('|')
        if v[GEO['SUMLEV'] - 1] == '160':
            places[v[GEO['PLACE'] - 1]] = v[86]  # BASENAME
        elif v[GEO['SUMLEV'] - 1] == '750' and v[GEO['COUNTY'] - 1] in COUNTIES:
            rows.append((v[GEO['GEOID'] - 1], v[GEO['COUNTY'] - 1], v[GEO['PLACE'] - 1], int(v[GEO['POP100'] - 1]),
                         float(v[GEO['INTPTLAT'] - 1]), float(v[GEO['INTPTLON'] - 1])))
blocks = pd.DataFrame(rows, columns=['geoid', 'county', 'place', 'pop', 'lat', 'lon'])
blocks['e'], blocks['n'] = grid.miles_from_origin(blocks['lat'].to_numpy(), blocks['lon'].to_numpy())
blocks['chicago'] = blocks['place'].eq(CHICAGO)
blocks['where'] = [places.get(p) or f'Unincorporated {COUNTIES[c]} County' for p, c in zip(blocks['place'], blocks['county'])]
P = int(blocks['pop'].sum())
log.info('%d blocks, %d residents', len(blocks), P)

## 3. Validate

In [ ]:
assert int(blocks.loc[blocks['chicago'], 'pop'].sum()) == 2_746_388, 'Chicago 2020 population'
assert P == 8_445_866, 'six-county 2020 population (Census: Cook 5,275,541; DuPage 932,877; Kane 516,522; Lake 714,342; McHenry 310,229; Will 696,355)'
for ag, f in FEEDS.items():
    assert f['trips'] > 500, f'{ag}: too few trips on {SERVICE_DAY}'
pd.DataFrame({k: {c: f[c] for c in ('snapshot', 'services', 'routes', 'trips')} for k, f in FEEDS.items()})

## 4. Service near each block

For each route, the most departures an hour at any one of its stops near the block, between 7:00
and 19:00; summed over routes. A route's busiest nearby stop sees it one way, so the sum reads as
"buses and trains an hour each way". Bus stops count within a quarter mile, 'L' and Metra stations
within a half mile, in a straight line.

In [ ]:
deps['rail'] = deps['route_type'].isin(['0', '1', '2'])
deps['key'] = deps['agency'] + ':' + deps['stop_id']
w = deps[(deps['h'] >= WINDOW[0]) & (deps['h'] < WINDOW[1])]
sr = deps.groupby(['key', 'agency', 'route_id', 'rail']).agg(e=('e', 'first'), n=('n', 'first'), last=('h', 'max')).reset_index()
sr = sr.merge(w.groupby(['key', 'route_id']).size().rename('n_win').reset_index(), how='left', on=['key', 'route_id'])
sr['tph'] = sr['n_win'].fillna(0) / (WINDOW[1] - WINDOW[0])

def service(XY: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # buses and trains an hour each way, and the last departure, near each point (miles)
    tph, last = np.zeros(len(XY)), np.full(len(XY), np.nan)
    for rail, rad in ((False, BUS_R), (True, RAIL_R)):
        s = sr[sr['rail'] == rail].reset_index(drop=True)
        pts = s[['e', 'n']].drop_duplicates().reset_index(drop=True).assign(pid=lambda d: np.arange(len(d)))
        s = s.merge(pts, on=['e', 'n'])
        near = cKDTree(pts[['e', 'n']].to_numpy()).query_ball_point(XY, rad)
        pairs = pd.DataFrame({'blk': np.repeat(np.arange(len(XY)), [len(x) for x in near]),
                              'pid': np.concatenate([np.asarray(x, int) for x in near])})
        del near
        j = pairs.merge(s[['pid', 'agency', 'route_id', 'tph', 'last']], on='pid')
        per = j.groupby(['blk', 'agency', 'route_id']).agg(tph=('tph', 'max'), last=('last', 'max')).reset_index()
        a = per.groupby('blk').agg(tph=('tph', 'sum'), last=('last', 'max'))
        tph += a['tph'].reindex(range(len(XY))).fillna(0).to_numpy()
        last = np.fmax(last, a['last'].reindex(range(len(XY))).to_numpy())
    return tph, last


blocks['tph'], blocks['last'] = service(blocks[['e', 'n']].to_numpy())
blocks['r'] = np.hypot(blocks['e'], blocks['n'])
pop = blocks['pop']
share = lambda m: float(pop[m].sum() / P)
log.info('every 15 min or better nearby: %.1f%%; nothing nearby: %.1f%%', 100 * share(blocks['tph'] >= 4), 100 * share(blocks['last'].isna()))

## 5. The edge of frequent service

The same count as for the blocks, at every point of a lattice a tenth of a mile apart over the map.
The edge is drawn round the points with four or more buses and trains an hour each way. Islands
and holes under half a square mile are left out of the drawing (a lone stop's quarter-mile circle);
every share in the text comes from the blocks, not from the drawing.

In [ ]:
W_, E_, S_, N_ = EXTENT
gx = np.arange(W_ + EDGE_STEP / 2, E_, EDGE_STEP)
gy = np.arange(S_ + EDGE_STEP / 2, N_, EDGE_STEP)
on = np.zeros((len(gy), len(gx)), bool)
for i0 in range(0, len(gy), 100):  # in bands, to keep memory low
    yy, xx = np.meshgrid(gy[i0:i0 + 100], gx, indexing='ij')
    t, _ = service(np.c_[xx.ravel(), yy.ravel()])
    on[i0:i0 + 100] = (t >= FREQ_TPH).reshape(xx.shape)
log.info('lattice: %d of %d points frequent', on.sum(), on.size)

# rows of the lattice as runs of rectangles, then one union
boxes = []
for i, row in enumerate(on):
    d = np.diff(np.r_[0, row.astype(int), 0])
    for a, b in zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1)):
        boxes.append(shapely.box(W_ + a * EDGE_STEP, S_ + i * EDGE_STEP, W_ + b * EDGE_STEP, S_ + (i + 1) * EDGE_STEP))
freq_geom = shapely.union_all(boxes, grid_size=0.001)


def drop_small(g, a_min):
    polys = g.geoms if isinstance(g, MultiPolygon) else [g]
    out = [Polygon(p.exterior, [h for h in p.interiors if Polygon(h).area >= a_min]) for p in polys if p.area >= a_min]
    return shapely.union_all(out)


freq_draw = drop_small(freq_geom, EDGE_MIN)
blocks['freq'] = blocks['tph'] >= FREQ_TPH
inside = shapely.contains_xy(freq_draw, blocks['e'].to_numpy(), blocks['n'].to_numpy())
agree = float(pop[inside == blocks['freq']].sum() / P)
log.info('frequent: %.1f%% of residents; the drawn edge agrees with the blocks for %.1f%%', 100 * share(blocks['freq']), 100 * agree)
assert agree > 0.97, 'the drawn edge should match the block count'

## 6. Analyze

Square miles of the address grid (floor of miles east and north of State & Madison). A square's
service is the resident-weighted mean of its blocks. Then the index from the Loop outward, the
equal-density comparison across the city limit, the inset and the finding.

In [ ]:
blocks['cx'] = np.floor(blocks['e']).astype(int)
blocks['cy'] = np.floor(blocks['n']).astype(int)
blocks['w'] = blocks['tph'] * pop
blocks['wl'] = blocks['last'].fillna(0) * pop
blocks['pl'] = pop.where(blocks['last'].notna(), 0)
blocks['pf'] = pop * blocks['freq']
top_place = blocks.groupby(['cx', 'cy', 'where'])['pop'].sum().reset_index().sort_values('pop').drop_duplicates(['cx', 'cy'], keep='last').set_index(['cx', 'cy'])['where']
cells = blocks.groupby(['cx', 'cy']).agg(pop=('pop', 'sum'), w=('w', 'sum'), chi=('chicago', 'mean'),
                                          wl=('wl', 'sum'), pl=('pl', 'sum'), pf=('pf', 'sum')).reset_index()
cells = cells[cells['pop'] >= 50].copy()
cells['tph'] = cells['w'] / cells['pop']
cells['last'] = np.where(cells['pl'] > 0, cells['wl'] / cells['pl'].clip(lower=1), np.nan)
cells['where'] = [top_place.get((x, y)) for x, y in zip(cells['cx'], cells['cy'])]
cells['r'] = np.hypot(cells['cx'] + 0.5, cells['cy'] + 0.5)
cells['ring'] = (np.floor(cells['r'] / 2) * 2).astype(int)
cells['city'] = np.where(cells['chi'] >= 0.5, 'chicago', 'suburb')
log.info('%d square miles with 50+ residents (%d residents)', len(cells), cells['pop'].sum())

# from the Loop outward: residents per populated square mile and service, by 2-mile ring, index 100 = first ring
rings_t = cells.groupby('ring').apply(lambda g: pd.Series({
    'pop': g['pop'].sum(), 'squares': len(g), 'density': g['pop'].sum() / len(g),
    'tph': np.average(g['tph'], weights=g['pop']), 'chicagoShare': g.loc[g['city'] == 'chicago', 'pop'].sum() / g['pop'].sum()}),
    include_groups=False).reset_index()
rings_t = rings_t[(rings_t['pop'] >= 20_000) & (rings_t['ring'] <= 46)].copy()
rings_t['dIdx'] = 100 * rings_t['density'] / rings_t['density'].iloc[0]
rings_t['sIdx'] = 100 * rings_t['tph'] / rings_t['tph'].iloc[0]
half_chicago = float(rings_t.loc[rings_t['chicagoShare'] < 0.5, 'ring'].iloc[0])
at25 = rings_t.loc[rings_t['ring'] == 24].iloc[0]
rings_t.round(2)

In [ ]:
# equal density: square miles with 5,000-10,000 residents, inside and outside Chicago
band = cells[(cells['pop'] >= 5000) & (cells['pop'] < 10000)]
eq = band.groupby('city').agg(squares=('pop', 'size'), residents=('pop', 'sum'), median_tph=('tph', 'median'))
dense_sub = cells[(cells['city'] == 'suburb') & (cells['pop'] >= 5000)]
thin = dense_sub[dense_sub['tph'] < 4]
summary_eq = {
    'chicagoSquares': int(eq.loc['chicago', 'squares']), 'suburbSquares': int(eq.loc['suburb', 'squares']),
    'chicagoMedian': round(float(eq.loc['chicago', 'median_tph']), 1), 'suburbMedian': round(float(eq.loc['suburb', 'median_tph']), 1),
    'denseSuburbSquares': len(dense_sub), 'denseSuburbResidents': int(dense_sub['pop'].sum()),
    'thinSquares': len(thin), 'thinResidents': int(thin['pop'].sum()),
}
eq, summary_eq

In [ ]:
# the inset: the near west, Chicago, Cicero and Berwyn along Cermak Road (2200 S)
INSET = {'west': -12, 'east': -2, 'south': -6, 'north': 2}  # miles
blocks['cellpop'] = blocks.groupby(['cx', 'cy'])['pop'].transform('sum')
ib = blocks[(blocks['e'] >= INSET['west']) & (blocks['e'] < INSET['east']) & (blocks['n'] >= INSET['south']) & (blocks['n'] < INSET['north'])]
ins = []
for name, m in (('Chicago', ib['chicago']), ('Cicero', ib['where'].eq('Cicero')), ('Berwyn', ib['where'].eq('Berwyn'))):
    g = ib[m]
    lab = cells[(cells['where'] == name) & cells['cx'].between(INSET['west'], INSET['east'] - 1) & cells['cy'].between(INSET['south'], INSET['north'] - 1)]
    ins.append({'name': name, 'residents': int(g['pop'].sum()),
                'density': round(float(np.average(g['cellpop'], weights=g['pop']))),
                'tph': round(float(np.average(g['tph'], weights=g['pop'])), 1),
                'x': round(float(np.average(lab['cx'] + 0.5, weights=lab['pop'])), 2), 'y': round(float(np.average(lab['cy'] + 0.5, weights=lab['pop'])), 2)})
inset = {'box': INSET, 'places': ins}
pd.DataFrame(ins)

In [ ]:
pct = lambda v: f'{round(100 * v)}%'
c_, s_ = summary_eq['chicagoMedian'], summary_eq['suburbMedian']
summary_f = {
    'frequentShare': round(share(blocks['freq']), 4),
    'frequentChicago': round(float(pop[blocks['chicago'] & blocks['freq']].sum() / pop[blocks['chicago']].sum()), 4),
    'frequentSuburbs': round(float(pop[~blocks['chicago'] & blocks['freq']].sum() / pop[~blocks['chicago']].sum()), 4),
    'frequentResidents': int(pop[blocks['freq']].sum()),
    'frequentSuburbResidents': int(pop[~blocks['chicago'] & blocks['freq']].sum()),
    'suburbResidents': int(pop[~blocks['chicago']].sum()),
}
fc_, fs_ = pct(summary_f['frequentChicago']), pct(summary_f['frequentSuburbs'])
FINDING = (f'Chicago’s grid of main streets continues past the city limit, but frequent bus and train service mostly '
           f'does not. Among square miles with 5,000 to 10,000 residents, half of those in Chicago get {c_:g} or more '
           f'buses and trains an hour, while half of those in the suburbs get {s_:g} or fewer. A bus or train comes at '
           f'least every 15 minutes for {fc_} of Chicago residents, but for only {fs_} of people in the suburbs.')
summary = {
    'residents': P, 'chicago': int(pop[blocks['chicago']].sum()),
    **summary_f,
    'noneShare': round(share(blocks['tph'] == 0), 4),
    'halfChicagoMile': half_chicago + 1,
    'density25': round(float(at25['dIdx']), 1), 'service25': round(float(at25['sIdx']), 1),
    **summary_eq,
}
print(FINDING)
summary

## 7. QA view

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 9))
for _, c in cells.iterrows():
    s = 0.92 * min(1, np.sqrt(c['pop'] / 15000))
    ax[0].add_patch(plt.Rectangle((c['cx'] + .5 - s / 2, c['cy'] + .5 - s / 2), s, s, color='0.75', lw=0))
    f = s * np.sqrt(min(1, c['tph'] / FULL_TPH))
    if f > 0.02:
        ax[0].add_patch(plt.Rectangle((c['cx'] + .5 - f / 2, c['cy'] + .5 - f / 2), f, f, color='0.1', lw=0))
for p in (freq_draw.geoms if isinstance(freq_draw, MultiPolygon) else [freq_draw]):
    for r in [p.exterior, *p.interiors]:
        ax[0].plot(*r.xy, color='k', lw=0.6)
ax[0].set(xlim=(-56, 6), ylim=(-48, 44), aspect=1, title='Square miles; edge of frequent service (4+ an hour)')
ax[1].plot(rings_t['ring'] + 1, rings_t['dIdx'], '--', color='0.5', label='residents per square mile')
ax[1].plot(rings_t['ring'] + 1, rings_t['sIdx'], color='0.1', label='buses and trains per hour')
ax[1].axvline(summary['halfChicagoMile'], color='0.6', lw=0.6)
ax[1].set(yscale='log', xlabel='miles from State & Madison', title='Index, first ring = 100'); ax[1].legend()
plt.show()

## 8. Export

`plate.json`: ledger, finding, summary, square miles (`cells`: cx, cy in miles; residents; service;
last departure; frequent share; place), the ring index, the frequent edge (address units), 'L' and Metra
track for the inset, and the inset. Keys are camelCase.

In [ ]:
def to_addr_rings(g, tol=0.02):
    g = g.simplify(tol)
    polys = g.geoms if isinstance(g, MultiPolygon) else [g]
    out = []
    for p in polys:
        for r in [p.exterior, *p.interiors]:
            a = np.asarray(r.coords)
            out.append([[round(float(x * 800)), round(float(grid.north_to_address(y)))] for x, y in a])
    return out


def track(ag: str, route_types: set[str], box: dict):
    t = trips[(trips['agency'] == ag) & trips['route_type'].isin(route_types)].drop_duplicates('shape_id')
    b = shapely.box(box['west'] - 1, box['south'] - 1, box['east'] + 1, box['north'] + 1)
    out = []
    for r in t.itertuples():
        ln = shapes.get((ag, r.shape_id))
        if ln is None or not ln.intersects(b):
            continue
        ln = ln.intersection(b).simplify(0.005)
        for part in getattr(ln, 'geoms', [ln]):
            if part.length > 0:
                out.append({'line': r.route_id, 'path': [[round(float(x * 800)), round(float(grid.north_to_address(y)))] for x, y in part.coords]})
    return out


CTA_LINE = {'Red': 'Red', 'Blue': 'Blue', 'Brn': 'Brown', 'G': 'Green', 'Org': 'Orange', 'P': 'Purple', 'Pink': 'Pink', 'Y': 'Yellow'}
rail = [{'line': CTA_LINE.get(t['line'], t['line']), 'path': t['path']} for t in track('cta', {'1'}, INSET)]
metra = [{'line': t['line'], 'path': t['path']} for t in track('metra', {'2'}, INSET)]

r1 = lambda v: None if pd.isna(v) else round(float(v), 2)
ledger = Ledger(
    source='CTA, Metra and Pace GTFS schedules',
    dataset_id='GTFS',
    url=FEEDS['cta']['url'],
    pulled=max(f['snapshot'] for f in FEEDS.values()),
    units=f'{P:,} residents (2020 Census) in {len(blocks):,} blocks of six counties; {len(cells):,} square miles',
    method=[
        f'Weekday service of {SERVICE_DAY:%A %-d %B %Y}, {WINDOW[0]}:00–{WINDOW[1]}:00: per route, departures an hour at its busiest stop within ¼ mi (bus) or ½ mi (rail), summed',
        'Square miles of Chicago’s address grid, extended past the city limit; service weighted by residents',
        f'Frequent: {FREQ_TPH} or more buses and trains an hour each way; edge drawn on a {EDGE_STEP} mi lattice, islands and holes under {EDGE_MIN} sq mi left out',
    ],
    caveats=[
        'Straight-line distance to a stop, not the walk',
        'Pace On Demand and Dial-a-Ride are booked services, not schedules, and are not counted; Indiana and Wisconsin are outside the six counties',
        '2020 population; one weekday',
    ],
)
payload = {
    'feeds': {k: {'name': f['name'], 'snapshot': f['snapshot'].isoformat(), 'url': f['url'], 'routes': f['routes'], 'trips': f['trips']} for k, f in FEEDS.items()},
    'serviceDay': SERVICE_DAY.isoformat(),
    'window': list(WINDOW),
    'fullTph': FULL_TPH,
    'freqTph': FREQ_TPH,
    'sources': {'census': PL_URL},
    'summary': summary,
    'cells': [
        {'x': int(c.cx), 'y': int(c.cy), 'pop': int(c.pop), 'tph': r1(c.tph), 'last': r1(c.last), 'where': c.where,
         'chicago': bool(c.city == 'chicago'), 'ring': int(c.ring),
         'freq': r1(c.pf / c.pop)}
        for c in cells.itertuples()
    ],
    'rings': [{'ring': int(r.ring), 'pop': int(r.pop), 'squares': int(r.squares), 'density': round(float(r.density)),
               'tph': r1(r.tph), 'dIdx': r1(r.dIdx), 'sIdx': r1(r.sIdx), 'chicagoShare': r1(r.chicagoShare)} for r in rings_t.itertuples()],
    'edge': to_addr_rings(freq_draw),
    'rail': rail,
    'metra': metra,
    'inset': inset,
}
path = write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload=payload)
assert 'NaN' not in path.read_text(), 'NaN is not valid JSON for the site'
log.info('Wrote %s (%.0f KB)', path.relative_to(DAY_DIR), path.stat().st_size / 1024)